# Video Game Commercial Success Prediction & Market Analysis

The goal of this project is to deliver a clear commercial narrative that game studios and publishers can use to minimize financial risk before greenlighting a new game's production.

When looking into creating new games studios and publishers investigate what has been historically successful. By analyzing characteristics of games that have already been released, we can help guide new game ideas to the right publishers to help propagate the game to a better sales pattern, allowing both the publisher and game developer to maximize their investment.

#### Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sqlalchemy import create_engine, text
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

## Step 1: Environment Setup & In-Memory Database Initialization (SQL)

To start, we will create an in-memory database using SQLAlchemy. 
By doing this, the data pipeline can be run anywhere.

In [ ]:
# Create an in memory SQLite database
engine = create_engine('sqlite:///:memory:')

In [ ]:
# Load data into pandas dataframe
df_raw = pd.read_csv('datasets/Video_Games_Sales.csv')
df_raw.info()

In [ ]:
# ingest raw data into the in-memory SQL database
df_raw.to_sql('raw_game_sales', con=engine, index=False, if_exists='replace')

In [ ]:
# Run SQL query directly against the database
query = """
SELECT count(*)
FROM raw_game_sales
"""

In [ ]:
with engine.connect() as conn:
    results = conn.execute(text(query))
    print(f'Successfully ingested {results.scalar()} records into in-memory table "raw_game_sales".')

## Step 2: Cleaning and Transformation (SQL)

Next we will pull the columns we need for the analysis. We will also filter out the `NULL` values from the `critic_scores`, `global_sales`, `year_of_release`, `user_score`, and `publisher`.

In [ ]:
# Query to pull only the needed columns from database
query = """
SELECT name as name,
    platform as platform,
    year_of_release as release_year,
    genre as genre, 
    publisher as publisher,
    na_sales as na_sales,
    eu_sales as eu_sales,
    jp_sales as jp_sales,
    other_sales as other_sales,
    global_sales as global_sales,
    critic_score as critic_score,
    user_score as user_score
FROM raw_game_sales
WHERE critic_score IS NOT NULL 
AND global_sales IS NOT NULL
AND year_of_release IS NOT NULL
AND user_score != 'tbd'
AND publisher IS NOT NULL;
"""

In [ ]:
with engine.connect() as conn:
    results = conn.execute(text(query))
    df_cleaned = pd.DataFrame(results)

In [ ]:
df_cleaned.info()

Looking above, we see that the `user_score` is of the type `object (string)`. We need to convert this to a numeric type to align the data.

In [ ]:
# Change column to numeric
df_cleaned['user_score'] = pd.to_numeric(df_cleaned['user_score'], errors='coerce')

In [ ]:
# Remove rows containing NaN values
df_cleaned = df_cleaned.dropna()

In [ ]:
# Confirm change
df_cleaned.info()

## Step 3: Exploratory Data Analysis & Feature Engineering (Python)

Now we need to set up the binary classification target: `Is_Hit` = `1` if `global_sales` $\geq 1.0$ else `0`.

In [ ]:
# create 'is_hit' column setting games that gloably made more than 1 million dollers to 1 and 0 if not
df_cleaned['is_hit'] = np.where(df_cleaned['global_sales'] >= 1.0, 1, 0)

In [ ]:
# confirm column creation and value placement
df_cleaned.sample(10)

Next we'll calculate the baseline class balance ratio (percetage of hits vs. non-hits).

In [ ]:
# Calulate the average and multiply by 100 to get the percentage
counts = df_cleaned['is_hit'].value_counts(normalize=True) * 100

print('Balance Ratios:')
print(f'hit:      {counts.get(1.0) :.2f}%')
print(f'not_hit: {counts.get(0.0) : .2f}%')

Above, we calculate the class balance ratios using `normalize=True`, then multiply by 100 to yield relative percentages rather than raw counts. By referencing values with explicit class labels (`1` for hit, `0` for non-hit) rather than positional indexing, the calculation remains accurate and robust to unexpected shifts in class distribution in future data ingestions.

#### Feature Selection & Preprocessing

In [ ]:
# column name reference
df_cleaned.columns

In [ ]:
# target vector
y = df_cleaned['is_hit']

# predictive features
features = ['platform','release_year','genre','publisher','critic_score','user_score']

X = df_cleaned[features]

> Note: The regional sales columns (`na_sales`, `eu_sales`, `jp_sales`, `other_sales`) and `global_sales` are retained in the cleaned SQL dataset for exploratory data analysis and regional marketing profiling. To prevent target leakage, all sales metrics will be strictly excluded from the feature matrix (`X`) prior to model training.

Next we will handle publishers with high-cardinality. There can be publishers with only 1 to 2 historical titles, which can introduce hundreds of dummy columns when encoded. Grouping these publishers into an `other` category will keep our feature matrix manageable.

In [ ]:
# keep top 20 publishers, group the rest as `other`
top_publishers = X['publisher'].value_counts().nlargest(20).index
X['publisher'] = X['publisher'].apply(lambda x: x if x in top_publishers else 'other')

X.sample(10)

#### Train, Test, Split

We run this initial `train_test_split` to help prevent data leakage. By splitting before encoding, we ensure that our test represents all the unseen data

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

#### Encoding

Here we will apply `pandas.get_dummies()` for quick exploration.

In [ ]:
# encoding features
categorical_columns = ['platform','genre','publisher']

# get dummy columns
X_encoded = pd.get_dummies(X, columns=categorical_columns,drop_first=True)

X_train_enc, X_test_enc, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42, stratify=y
)

#### Exploratory Analysis

Since we've calculated the baseline target class balance(`is_hit` ratio of $81/19$). We will generate some visual plots to verify the features belong in our model.

#### Correlation Matrix & Feature Distribution Plots

In [ ]:
# numeric correlation matrix
plt.figure(figsize=(10,6))
numeric_cols = ['critic_score','user_score','na_sales','eu_sales','jp_sales','other_sales','global_sales']
sns.heatmap(df_cleaned[numeric_cols].corr(), annot=True, cmap='Blues',fmt='.2f')
plt.title('Feature Correlation Matrix')
plt.show()

# score distribution plot
fig, axes = plt.subplots(1,2, figsize=(12,5))
sns.histplot(df_cleaned['critic_score'], kde=True, ax=axes[0], color='skyblue').set(title='Critic Score Distribution')
sns.histplot(df_cleaned['user_score'], kde=True, ax=axes[1], color='salmon').set(title='User Score Distribution')
plt.tight_layout()
plt.show()

####  Regional Sales Summary Metrics

In [ ]:
# regional sales summary statistics
regional_cols = ['na_sales','eu_sales','jp_sales','other_sales','global_sales']
regional_summary = df_cleaned[regional_cols].describe().T[['mean','std','min','50%','max']]
regional_summary.columns = ['Mean ($M)', 'Std Dev', 'Min', 'Median', 'Max']

display(regional_summary)

Across all regions, the sales data exhibits high right-skewness. The `Median` regional sales are significantly lower than the `mean` sales. This confirms that a small percentage of blockbuster titles accounts for the vast majority of commercial volume.

North American sales (`na_sales`) demonstrate the strongest individual regional correlation with global success (`global_sales`), averaging $\$0.39M$ per title. European sales (`eu_sales`) follow closely as the second largest contributor.

Japanese sales (`jp_sales`) show a significantly lower correlation with Western sales performance. This highlights distinct regional consumer preferences: titles that perform exceptionally well in Western markets do not automatically replicate that success in Japan, and vice versa.

`critic score` demonstrates a stronger positive correlation with `global_sales` than `user_score`. This indicates that professional critic evaluations serve as a stronger leading indicator of commercial viability during a title's initial launch window than consumer review scores.

### Step 3: Predictive Modeling & Hypothesis Testing (Python)

For the model, I've chosen to use `LogisticRegression` to predict commercial hits.

The target metric (`is_hit`) is binary-categorized as `1` for games with global sales $\ge \$1.0\text{M}$ and `0`. Logistic Regression uses the sigmoid function to map feature inputs (e.g., `scores`, `genre`, `platform`, `publisher`) directly into probabilities between `0` and `1`, which helps avoid the out-of-bounds prediction errors inherent to linear regression models.

Unlike 'Black box'  machine learning models, Logistic regression provides clear, quantifiable feature coefficients ($\beta$). Converting these coefficients to **Odd Ratios** ($e^{\beta}$) allows stakeholders to measure exactly how individual factors, such as a 10-point bump in review scores or release on a specific console, increase or decrease a game's likelihood of becoming a commercial success.

In [ ]:
# Scale continuous numerical features
from sklearn.preprocessing import StandardScaler
scalar = StandardScaler()

# scale only continuous numerical columns
X_train_enc[['critic_score','user_score']] = scalar.fit_transform(X_train_enc[['critic_score','user_score']])
X_test_enc[['critic_score', 'user_score']] = scalar.transform(X_test_enc[['critic_score','user_score']])

In [ ]:
# create model
model = LogisticRegression(max_iter=10000, random_state=42)

In [ ]:
# fit model
model_fit = model.fit(X_train_enc, y_train)

In [ ]:
# check model accuracy
acc = accuracy_score(y_test,model_fit.predict(X_test_enc)) * 100

In [ ]:
print(f'Logistic Regression model accuaracy: {acc:.2f}%')